# 14 · County Foundation
**Session 0.1 of the TERRA County App roadmap.**
Implements the five-scale doctrine: county (analysis/governance) · bus (energy
intervention) · tract (social measurement) · ecoregion (ecological suitability)
· material ledger (honesty).

**Do not modify `src/terra_engine.py` in this session.**


## 0 · Setup

In [1]:

import json, os, warnings, io, time
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import mapping
import requests
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# Resolve project root whether run from project root or notebooks/ subdir
_here = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
BASE  = _here.parent if _here.name == 'notebooks' else _here
load_dotenv(BASE / '.env')

PROC      = BASE / 'data/processed'
RAW       = BASE / 'data/raw'
CENSUS_KEY = os.environ.get('CENSUS_API_KEY', '')
BEA_KEY    = os.environ.get('BEA_API_KEY', '')   # may be absent

# Study state FIPS → abbreviation
STUDY_STATE_FIPS = {
    '56': 'WY', '08': 'CO', '30': 'MT',
    '49': 'UT', '16': 'ID', '31': 'NE', '46': 'SD',
}

# Six study ecoregions (US_L3NAME in mw_ecoregions.geojson)
SIX_ECO_NAMES = {
    'Wyoming Basin', 'Northwestern Great Plains', 'Middle Rockies',
    'Southern Rockies', 'Colorado Plateaus', 'High Plains',
}

print("Environment ready.")
print(f"  CENSUS_API_KEY present: {bool(CENSUS_KEY)}")
print(f"  BEA_API_KEY present:    {bool(BEA_KEY)}")


Environment ready.
  CENSUS_API_KEY present: True
  BEA_API_KEY present:    False


In [2]:

# ── Load all spatial inputs & print summary table ────────────────────────────
ees      = pd.read_parquet(PROC / 'mw_tract_ees_scores.parquet')
buses    = gpd.read_file(PROC / 'synthetic_buses.geojson')
eco      = gpd.read_file(PROC / 'mw_ecoregions.geojson')
tiger    = gpd.read_file(RAW  / 'census_counties/tl_2023_us_county.shp')

def bb(gdf):
    b = gdf.total_bounds
    return f"({b[0]:.2f}, {b[1]:.2f}, {b[2]:.2f}, {b[3]:.2f})"

rows = [
    ('mw_tract_ees_scores.parquet', len(ees),    'no geometry',  'N/A'),
    ('synthetic_buses.geojson',     len(buses),   str(buses.crs), bb(buses)),
    ('mw_ecoregions.geojson',       len(eco),     str(eco.crs),   bb(eco)),
    ('tl_2023_us_county.shp',       len(tiger),   str(tiger.crs), bb(tiger)),
]
print(f"{'File':<40} {'N':>6}  {'CRS':<15}  Bounding Box")
print('-'*105)
for f, n, c, b in rows:
    print(f"{f:<40} {n:>6}  {c:<15}  {b}")

print()
print("CRS check:")
print("  buses      EPSG:4326 ✓")
print("  ecoregions EPSG:4326 ✓")
print("  TIGER      EPSG:4269 (NAD83) → will reproject to EPSG:4326")


File                                          N  CRS              Bounding Box
---------------------------------------------------------------------------------------------------------
mw_tract_ees_scores.parquet                1676  no geometry      N/A
synthetic_buses.geojson                     500  EPSG:4326        (-156.33, 20.25, -63.39, 63.36)
mw_ecoregions.geojson                       128  EPSG:4326        (-115.00, 36.00, -100.00, 49.00)
tl_2023_us_county.shp                      3235  EPSG:4269        (-179.23, -14.60, 179.86, 71.44)

CRS check:
  buses      EPSG:4326 ✓
  ecoregions EPSG:4326 ✓
  TIGER      EPSG:4269 (NAD83) → will reproject to EPSG:4326


---
## 1 · Study Area Definition

All 23 WY counties + any county in CO / MT / UT / ID / NE / SD whose
**centroid** falls inside the union of the six study ecoregions.


In [3]:

# 1a. Build union of six study ecoregions
six_eco = eco[eco['US_L3NAME'].isin(SIX_ECO_NAMES)].copy()
print(f"Ecoregion features used: {len(six_eco)}")
print("Names:", sorted(six_eco['US_L3NAME'].unique()))

eco_union = unary_union(six_eco.geometry)
print(f"\nEcoregion union area (deg²): {eco_union.area:.2f}")


Ecoregion features used: 64
Names: ['Colorado Plateaus', 'High Plains', 'Middle Rockies', 'Northwestern Great Plains', 'Southern Rockies', 'Wyoming Basin']



Ecoregion union area (deg²): 121.09


In [4]:

# 1b. Load TIGER, reproject EPSG:4269 → EPSG:4326, filter to study states
tiger_4326 = tiger.to_crs('EPSG:4326')
study_tiger = tiger_4326[tiger_4326['STATEFP'].isin(STUDY_STATE_FIPS.keys())].copy()
study_tiger['state_abbr'] = study_tiger['STATEFP'].map(STUDY_STATE_FIPS)
print(f"TIGER counties in study states: {len(study_tiger)}")
for s, grp in study_tiger.groupby('state_abbr'):
    print(f"  {s}: {len(grp)}")


TIGER counties in study states: 375
  CO: 64
  ID: 44
  MT: 56
  NE: 93
  SD: 66
  UT: 29
  WY: 23


In [5]:

# 1c. Compute centroids in equal-area CRS (EPSG:5070) for accuracy,
#     then test membership in ecoregion union (EPSG:4326).
#     Area also computed in EPSG:5070 (metre-based).
ea = study_tiger.to_crs('EPSG:5070')
study_tiger['area_km2']     = (ea.geometry.area / 1e6).round(2)
study_tiger['centroid_lat'] = study_tiger.geometry.centroid.y.round(6)
study_tiger['centroid_lon'] = study_tiger.geometry.centroid.x.round(6)

# Test centroid-in-ecoregion-union for non-WY counties
from shapely.geometry import Point

def centroid_in_union(row):
    if row['state_abbr'] == 'WY':
        return True   # all 23 WY counties always included
    pt = Point(row['centroid_lon'], row['centroid_lat'])
    return eco_union.contains(pt)

study_tiger['in_study'] = study_tiger.apply(centroid_in_union, axis=1)

study_counties = study_tiger[study_tiger['in_study']].copy()
print(f"Study counties selected: {len(study_counties)}")
for s, grp in study_counties.groupby('state_abbr'):
    print(f"  {s}: {len(grp)}")

wy_count = (study_counties['state_abbr'] == 'WY').sum()
assert wy_count == 23, f"Expected 23 WY counties, got {wy_count}"
print(f"\n✓ WY county count == 23")
n = len(study_counties)
if 90 <= n <= 120:
    print(f"✓ Total study counties ({n}) in roadmap estimate 90–120")
else:
    print(f"⚠ NOTE: Study county count = {n}, outside roadmap estimate of 90–120.")
    print("  This is expected: the six study ecoregions (especially High Plains,")
    print("  Middle Rockies, Northwestern Great Plains) have large spatial extents")
    print("  that include most of CO (49/64 counties) and MT (37/56 counties).")
    print("  The centroid-in-union test is correct; the 90-120 estimate was conservative.")
    print("  Proceeding with all", n, "counties.")


Study counties selected: 157
  CO: 49
  ID: 6
  MT: 37
  NE: 13
  SD: 19
  UT: 10
  WY: 23

✓ WY county count == 23
⚠ NOTE: Study county count = 157, outside roadmap estimate of 90–120.
  This is expected: the six study ecoregions (especially High Plains,
  Middle Rockies, Northwestern Great Plains) have large spatial extents
  that include most of CO (49/64 counties) and MT (37/56 counties).
  The centroid-in-union test is correct; the 90-120 estimate was conservative.
  Proceeding with all 157 counties.


In [6]:

# 1d. Save mw_study_counties.csv
study_counties['in_wyoming'] = study_counties['state_abbr'] == 'WY'

csv_out = study_counties[['GEOID','NAME','state_abbr','in_wyoming',
                            'centroid_lon','centroid_lat','area_km2']].rename(
    columns={'NAME': 'county_name', 'state_abbr': 'state'})
csv_out = csv_out.sort_values(['state','county_name']).reset_index(drop=True)

csv_out.to_csv(PROC / 'mw_study_counties.csv', index=False)
print(f"Saved mw_study_counties.csv — {len(csv_out)} rows")
print(csv_out.head(5).to_string())


Saved mw_study_counties.csv — 157 rows
   GEOID county_name state  in_wyoming  centroid_lon  centroid_lat  area_km2
0  08001       Adams    CO       False   -104.337896     39.873650   3065.54
1  08005    Arapahoe    CO       False   -104.339276     39.649773   2086.31
2  08007   Archuleta    CO       False   -107.048287     37.193610   3510.45
3  08009        Baca    CO       False   -102.560480     37.319179   6623.54
4  08013     Boulder    CO       False   -105.357705     40.092467   1917.81


In [7]:

# 1e. Topology-preserving simplification for browser delivery
#     Target: ≤50 KB per state when subset extracted as GeoJSON.
#     Iteratively increase tolerance until all states pass.

import json as _json

def state_geojson_kb(gdf_state):
    """Return approximate GeoJSON size for a state subset (KB)."""
    geojson_str = gdf_state.to_json()
    return len(geojson_str.encode('utf-8')) / 1024

simplified = study_counties[['GEOID','NAME','state_abbr','geometry']].copy()

tol = 0.005   # start at ~500 m in degrees
for attempt in range(8):
    simp = simplified.copy()
    simp['geometry'] = simp.geometry.simplify(tol, preserve_topology=True)
    sizes = {}
    ok = True
    for state, grp in simp.groupby('state_abbr'):
        kb = state_geojson_kb(grp)
        sizes[state] = kb
        if kb > 50:
            ok = False
    if ok:
        break
    tol = round(tol + 0.005, 3)
    print(f"  tolerance {tol-0.005:.3f}° → some states > 50KB; retrying at {tol:.3f}°")

print(f"\nFinal simplification tolerance: {tol:.3f}°  (preserve_topology=True)")
print("State GeoJSON sizes (KB):")
for state in sorted(sizes):
    flag = "✓" if sizes[state] <= 50 else "✗ OVER"
    print(f"  {state}: {sizes[state]:.1f} KB  {flag}")

# Save
geojson_cols = simp[['GEOID','NAME','state_abbr','geometry']].rename(
    columns={'NAME': 'county_name', 'state_abbr': 'state'})
geojson_cols.to_file(PROC / 'mw_counties.geojson', driver='GeoJSON')
total_kb = (PROC / 'mw_counties.geojson').stat().st_size / 1024
print(f"\nSaved mw_counties.geojson — total {total_kb:.1f} KB, {len(geojson_cols)} features")


  tolerance 0.005° → some states > 50KB; retrying at 0.010°


  tolerance 0.010° → some states > 50KB; retrying at 0.015°


  tolerance 0.015° → some states > 50KB; retrying at 0.020°



Final simplification tolerance: 0.020°  (preserve_topology=True)
State GeoJSON sizes (KB):
  CO: 40.9 KB  ✓
  ID: 9.8 KB  ✓
  MT: 49.5 KB  ✓
  NE: 5.3 KB  ✓
  SD: 11.6 KB  ✓
  UT: 12.1 KB  ✓
  WY: 17.4 KB  ✓

Saved mw_counties.geojson — total 157.0 KB, 157 features


---
## 2 · Four-Way Crosswalk

One row per **(geoid, bus_id, ecoregion_code)** intersection.
`bus_weight` = inverse-distance share across 3 nearest in-study buses (sums to 1 per county).
`ecoregion_area_share` = fraction of county area in each ecoregion.


In [8]:

# 2a. Filter buses to Mountain West study area
#     Use the ecoregion bounding box + 2° buffer as the filter polygon.
from shapely.geometry import box as shapely_box

mw_bbox = shapely_box(-117, 34, -98, 51)
study_buses = buses[buses.geometry.within(mw_bbox)].copy()
print(f"In-study buses (within extended MW bbox): {len(study_buses)}")
print("BA codes:", sorted(study_buses['ba_code'].unique()))


In-study buses (within extended MW bbox): 75
BA codes: ['AESO', 'AZPS', 'BPAT', 'CISO', 'GRIF', 'GWA', 'IPCO', 'MISO', 'MPCO', 'NEVP', 'NWMT', 'PACE', 'PNM', 'PSCO', 'SPC', 'SRP', 'SWPP', 'WACM', 'WALC', 'WAUW', 'WWA']


In [9]:

# 2b. For each study county, find 3 nearest in-study buses.
#     Distances computed from county centroid (lat/lon) to bus location.
from scipy.spatial import cKDTree

bus_coords = np.column_stack([study_buses['lon'].values, study_buses['lat'].values])
bus_ids    = study_buses['bus_id'].values

county_lons = csv_out['centroid_lon'].values
county_lats = csv_out['centroid_lat'].values
county_coords = np.column_stack([county_lons, county_lats])
county_geoids = csv_out['GEOID'].values

tree = cKDTree(bus_coords)
K = 3
dists, idxs = tree.query(county_coords, k=K)

# Inverse-distance weights (handle exact coincidence)
EPS = 1e-6
weights = 1.0 / (dists + EPS)
weights = weights / weights.sum(axis=1, keepdims=True)  # normalise per county

bus_rows = []
for i, geoid in enumerate(county_geoids):
    for k in range(K):
        bus_rows.append({
            'geoid':       geoid,
            'bus_id':      int(bus_ids[idxs[i, k]]),
            'bus_weight':  round(float(weights[i, k]), 8),
            'primary_bus': k == 0,
            'dist_km':     round(float(dists[i, k]) * 111.32, 2),
        })

bus_df = pd.DataFrame(bus_rows)
print(f"Bus assignment rows: {len(bus_df)}  (= {len(csv_out)} counties × {K} buses)")
print(bus_df.head(6).to_string())


Bus assignment rows: 471  (= 157 counties × 3 buses)
   geoid  bus_id  bus_weight  primary_bus  dist_km
0  08001     488    0.994077         True     0.05
1  08001     380    0.005246        False     9.81
2  08001     376    0.000677        False    76.01
3  08005     380    0.560465         True    15.11
4  08005     488    0.339172        False    24.97
5  08005     377    0.100363        False    84.40


In [10]:

# 2c. Ecoregion area shares per county — exact polygon intersection
#     Use equal-area CRS for accurate areas.
study_counties_ea = study_counties[['GEOID','geometry']].to_crs('EPSG:5070')
eco_sub_ea = six_eco[['US_L3CODE','US_L3NAME','geometry']].to_crs('EPSG:5070')

print("Computing ecoregion × county intersections (may take ~30s)...")
t0 = time.time()
eco_xwalk = gpd.overlay(
    study_counties_ea.rename(columns={'GEOID':'geoid'}),
    eco_sub_ea.rename(columns={'US_L3CODE':'ecoregion_code','US_L3NAME':'ecoregion_name'}),
    how='intersection', keep_geom_type=False
)
eco_xwalk['intersection_area_m2'] = eco_xwalk.geometry.area
print(f"  done in {time.time()-t0:.1f}s — {len(eco_xwalk)} intersection fragments")

# county total area
county_areas = study_counties_ea.set_index('GEOID').geometry.area.rename('county_area_m2')
eco_xwalk = eco_xwalk.join(county_areas, on='geoid')
eco_xwalk['ecoregion_area_share'] = (
    eco_xwalk['intersection_area_m2'] / eco_xwalk['county_area_m2']
).round(6)

# Drop negligible slivers (<0.5% area)
eco_xwalk = eco_xwalk[eco_xwalk['ecoregion_area_share'] >= 0.005].copy()

# Renormalise shares per county to sum to 1
county_share_sum = eco_xwalk.groupby('geoid')['ecoregion_area_share'].sum()
eco_xwalk = eco_xwalk.join(county_share_sum.rename('share_sum'), on='geoid')
eco_xwalk['ecoregion_area_share'] = (
    eco_xwalk['ecoregion_area_share'] / eco_xwalk['share_sum']
).round(6)

eco_clean = eco_xwalk[['geoid','ecoregion_code','ecoregion_name','ecoregion_area_share']].copy()
print(f"Ecoregion rows after deduplication: {len(eco_clean)}")
print(eco_clean.head(6).to_string())


Computing ecoregion × county intersections (may take ~30s)...


  done in 0.5s — 380 intersection fragments
Ecoregion rows after deduplication: 242
    geoid ecoregion_code    ecoregion_name  ecoregion_area_share
0   08109             21  Southern Rockies              1.000000
1   49033             18     Wyoming Basin              1.000000
6   56023             18     Wyoming Basin              0.528008
8   56023             17    Middle Rockies              0.471992
9   31101             25       High Plains              1.000000
10  56013             18     Wyoming Basin              0.767967


In [11]:

# 2d. Build four-way crosswalk: (geoid, bus_id, ecoregion_code)
#     Cross-join bus rows × ecoregion rows per county.
crosswalk = bus_df.merge(eco_clean, on='geoid', how='left')

# Assign a default ecoregion for counties with no ecoregion overlap
# (counties fully outside the 6 study ecoregions — e.g. peripheral WY counties)
no_eco = crosswalk['ecoregion_code'].isna()
if no_eco.any():
    n_no_eco = crosswalk.loc[no_eco, 'geoid'].nunique()
    print(f"  WARNING: {n_no_eco} counties have no ecoregion overlap — assigning 'NONE'")
    crosswalk.loc[no_eco, 'ecoregion_code'] = 'NONE'
    crosswalk.loc[no_eco, 'ecoregion_name'] = 'None'
    crosswalk.loc[no_eco, 'ecoregion_area_share'] = 0.0

crosswalk = crosswalk[[
    'geoid','bus_id','bus_weight','ecoregion_code','ecoregion_area_share','primary_bus'
]].copy()

print(f"Crosswalk shape: {crosswalk.shape}")
print(crosswalk.head(8).to_string())


Crosswalk shape: (726, 6)
   geoid  bus_id  bus_weight ecoregion_code  ecoregion_area_share  primary_bus
0  08001     488    0.994077             25              1.000000         True
1  08001     380    0.005246             25              1.000000        False
2  08001     376    0.000677             25              1.000000        False
3  08005     380    0.560465             25              1.000000         True
4  08005     488    0.339172             25              1.000000        False
5  08005     377    0.100363             25              1.000000        False
6  08007     487    0.350674             21              0.993667         True
7  08007     487    0.350674             20              0.006333         True


In [12]:

# 2e. VALIDATION — fail loudly
print("=" * 60)
print("CROSSWALK VALIDATION")
print("=" * 60)

# 1. Every study county has >= 1 bus
counties_in_xwalk = set(crosswalk['geoid'].unique())
all_study_geoids  = set(csv_out['GEOID'].unique())
missing_buses = all_study_geoids - counties_in_xwalk
assert len(missing_buses) == 0, f"Counties with no bus assignment: {missing_buses}"
print(f"✓ All {len(all_study_geoids)} study counties have ≥1 bus")

# 2. Per-county bus_weight sums == 1.0 within 1e-6
weight_sums = (
    crosswalk.drop_duplicates(['geoid','bus_id'])
             .groupby('geoid')['bus_weight'].sum()
)
bad = weight_sums[np.abs(weight_sums - 1.0) > 1e-6]
assert len(bad) == 0, f"Bus weights don't sum to 1 for: {bad.index.tolist()}"
print(f"✓ Per-county bus_weight sums == 1.0 (±1e-6) for all {len(weight_sums)} counties")

# 3. Every geoid in crosswalk appears in mw_study_counties.csv
extra = counties_in_xwalk - all_study_geoids
assert len(extra) == 0, f"GEOIDs in crosswalk but not in study counties: {extra}"
print(f"✓ All crosswalk GEOIDs appear in mw_study_counties.csv")

print("=" * 60)
print("ALL CROSSWALK VALIDATIONS PASSED")
print("=" * 60)

# Save
crosswalk.to_parquet(PROC / 'county_crosswalk.parquet', index=False)
print(f"\nSaved county_crosswalk.parquet — {len(crosswalk)} rows")

# Summary stats
n_multi_bus_counties = (
    crosswalk.drop_duplicates(['geoid','bus_id'])
             .groupby('geoid')['bus_id'].nunique()
    > 1
).sum()
mean_buses = crosswalk.drop_duplicates(['geoid','bus_id']).groupby('geoid')['bus_id'].nunique().mean()
print(f"Counties with multiple buses: {n_multi_bus_counties}")
print(f"Mean buses per county: {mean_buses:.2f}")


CROSSWALK VALIDATION
✓ All 157 study counties have ≥1 bus
✓ Per-county bus_weight sums == 1.0 (±1e-6) for all 157 counties
✓ All crosswalk GEOIDs appear in mw_study_counties.csv
ALL CROSSWALK VALIDATIONS PASSED

Saved county_crosswalk.parquet — 726 rows
Counties with multiple buses: 157
Mean buses per county: 3.00


---
## 3 · EES Re-aggregation: Tract → County

Tract→county is an **exact nesting**: use GEOID prefix match (first 5 characters),
not spatial join.  Population-weighted mean of E, Ec, S scores per county.


In [13]:

# 3a. Assign county GEOID from first 5 chars of tract GEOID
ees['county_geoid'] = ees['GEOID'].str[:5]

# Filter to study counties only
ees_study = ees[ees['county_geoid'].isin(all_study_geoids)].copy()
print(f"Tracts in study area: {len(ees_study)} of {len(ees)} total")
print(f"Unique county GEOIDs covered: {ees_study['county_geoid'].nunique()}")
print(f"Study counties with ≥1 tract: {ees_study['county_geoid'].nunique()} "
      f"of {len(all_study_geoids)}")


Tracts in study area: 1566 of 1676 total
Unique county GEOIDs covered: 157
Study counties with ≥1 tract: 157 of 157


In [14]:

# 3b. Population-weighted mean per county
def pop_wmean(grp, col):
    pop = grp['population'].fillna(0)
    if pop.sum() == 0:
        return grp[col].mean()
    return (grp[col] * pop).sum() / pop.sum()

records = []
for geoid, grp in ees_study.groupby('county_geoid'):
    records.append({
        'geoid':       geoid,
        'E':           round(pop_wmean(grp, 'E_score'),  4),
        'Ec':          round(pop_wmean(grp, 'Ec_score'), 4),
        'S':           round(pop_wmean(grp, 'S_score'),  4),
        'population':  int(grp['population'].fillna(0).sum()),
        'n_tracts':    len(grp),
    })

county_ees = pd.DataFrame(records)

# Merge county name
name_map = csv_out.set_index('GEOID')['county_name']
state_map = csv_out.set_index('GEOID')['state']
county_ees['county_name'] = county_ees['geoid'].map(name_map)
county_ees['state']       = county_ees['geoid'].map(state_map)
county_ees = county_ees[['geoid','county_name','state','E','Ec','S','population','n_tracts']]

print(f"County EES summary: {len(county_ees)} counties")
print(county_ees.describe()[['E','Ec','S']].to_string())


County EES summary: 157 counties
                E          Ec           S
count  157.000000  157.000000  157.000000
mean     4.055709    5.609414    4.751418
std      2.404964    0.848964    0.578938
min      0.435100    2.829600    2.588100
25%      2.726300    5.141200    4.485500
50%      3.289500    5.598700    4.842300
75%      6.411900    6.021000    5.105600
max      9.407400    8.531000    5.944300


In [15]:

# 3c. Comparison table: old ecoregion-level means vs new county-pop-weighted means
#     Load ecoregion baseline scores from network_metadata.json
with open(PROC / 'network_metadata.json') as f:
    meta = json.load(f)

eco_scores = meta['ees_baseline']['ecoregion_scores']
# Population totals from mw_ecoregion_ees_summary.csv
eco_sum_df = pd.read_csv(PROC / 'mw_ecoregion_ees_summary.csv')

# Old study-area mean: population-weighted across the 7 study ecoregions
old_records = []
for code_str, info in eco_scores.items():
    pop_row = eco_sum_df[eco_sum_df['ecoregion_code'].astype(str) == code_str]
    pop = float(pop_row['population_total'].iloc[0]) if len(pop_row) else 0
    old_records.append({
        'ecoregion': info['name'],
        'code': code_str,
        'E': info['E_score'], 'Ec': info['Ec_score'], 'S': info['S_score'],
        'pop': pop,
    })
old_df = pd.DataFrame(old_records)
old_E  = (old_df['E'] * old_df['pop']).sum() / old_df['pop'].sum()
old_Ec = (old_df['Ec'] * old_df['pop']).sum() / old_df['pop'].sum()
old_S  = (old_df['S'] * old_df['pop']).sum() / old_df['pop'].sum()

# New study-area mean: population-weighted across study counties
new_E  = (county_ees['E'] * county_ees['population']).sum() / county_ees['population'].sum()
new_Ec = (county_ees['Ec'] * county_ees['population']).sum() / county_ees['population'].sum()
new_S  = (county_ees['S'] * county_ees['population']).sum() / county_ees['population'].sum()

print("=" * 65)
print("EES COMPARISON: Ecoregion-level (old) vs County-pop-weighted (new)")
print("=" * 65)
print(f"{'Capital':<8} {'Old (ecoregion)':>18} {'New (county)':>14} {'Delta':>8} {'Flag':>6}")
print("-" * 65)
for cap, old_v, new_v in [('E', old_E, new_E), ('Ec', old_Ec, new_Ec), ('S', old_S, new_S)]:
    delta = new_v - old_v
    flag = '  ✓' if abs(delta) <= 0.3 else '  ⚠ INVESTIGATE'
    print(f"{cap:<8} {old_v:>18.4f} {new_v:>14.4f} {delta:>+8.4f} {flag}")
print("=" * 65)

# Fail loudly if any capital deviates > 0.3
for cap, old_v, new_v in [('E', old_E, new_E), ('Ec', old_Ec, new_Ec), ('S', old_S, new_S)]:
    delta = abs(new_v - old_v)
    assert delta <= 0.3, (
        f"STOP: Capital {cap} deviates by {delta:.4f} (>0.3).\n"
        f"  Old (ecoregion-weighted): {old_v:.4f}\n"
        f"  New (county-weighted):    {new_v:.4f}\n"
        f"  Possible causes: (1) new study area includes non-ecoregion counties whose "
        f"tracts pull the mean; (2) some WY counties have no tract records in the EES "
        f"parquet; (3) population weighting base changed. Investigate before proceeding."
    )
print("All capitals within ±0.3 tolerance. Proceeding.")


EES COMPARISON: Ecoregion-level (old) vs County-pop-weighted (new)
Capital     Old (ecoregion)   New (county)    Delta   Flag
-----------------------------------------------------------------
E                    3.1481         3.1098  -0.0383   ✓
Ec                   6.9195         6.9322  +0.0126   ✓
S                    5.2908         5.2827  -0.0081   ✓
All capitals within ±0.3 tolerance. Proceeding.


In [16]:

# 3d. Save mw_county_ees_summary.csv
save_cols = ['geoid','county_name','E','Ec','S','population','n_tracts']
county_ees[save_cols].to_csv(PROC / 'mw_county_ees_summary.csv', index=False)
print(f"Saved mw_county_ees_summary.csv — {len(county_ees)} rows")
print(county_ees[save_cols].head(8).to_string())


Saved mw_county_ees_summary.csv — 157 rows
   geoid county_name       E      Ec       S  population  n_tracts
0  08001       Adams  2.3531  8.0292  5.2817      520149       106
1  08005    Arapahoe  2.3825  7.5176  5.5025      654453       161
2  08007   Archuleta  8.1814  5.0338  5.2356       13509         5
3  08009        Baca  2.9803  5.3880  4.4717        3496         2
4  08013     Boulder  2.7902  7.9611  5.7646      328658        78
5  08014  Broomfield  2.3426  8.5310  5.9443       73946        21
6  08015     Chaffee  7.3549  5.6342  5.0338       19564         6
7  08017    Cheyenne  3.1282  5.6268  4.7045        1726         1


---
## 4 · County Baseline Cards

Assemble `mw_county_cards.json` — one record per county GEOID.
Data sources (in priority order):
1. **Census ACS 5-year 2022** — population, median_hh_income, employment proxies
2. **BEA CAINC30** — employment, per_capita_income (if API key available; else ACS proxy)
3. **EIA-860 / power_plants_with_ba.geojson** — generation capacity, fuel mix
4. **USGS 2015 county water use** — total withdrawals (MGD)
5. **Hand-curated** — flagship Wyoming energy assets


In [17]:

# 4a. Census ACS 5-year 2022 — county-level demographics
#     Variables:
#       B01003_001E = total population
#       B19013_001E = median household income
#       B19301_001E = per capita income
#       B23025_004E = civilian employed (16+)
#       B23025_003E = civilian in labor force (16+)

ACS_VARS = ['B01003_001E','B19013_001E','B19301_001E','B23025_004E','B23025_003E']
ACS_YEAR = 2022

def fetch_acs_county_state(state_fips, variables=ACS_VARS, year=ACS_YEAR):
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        'get': 'NAME,' + ','.join(variables),
        'for': 'county:*',
        'in':  f'state:{state_fips}',
        'key': CENSUS_KEY,
    }
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        df = pd.DataFrame(data[1:], columns=data[0])
        df['GEOID'] = df['state'] + df['county']
        return df
    except Exception as e:
        print(f"  ACS fetch failed for state {state_fips}: {e}")
        return None

if CENSUS_KEY:
    acs_parts = []
    for fips, abbr in STUDY_STATE_FIPS.items():
        print(f"  Fetching ACS for {abbr} ({fips})...", end=' ')
        df = fetch_acs_county_state(fips)
        if df is not None:
            acs_parts.append(df)
            print(f"{len(df)} counties")
        else:
            print("FAILED")

    if acs_parts:
        acs_county = pd.concat(acs_parts, ignore_index=True)
        # Keep only study counties
        acs_county = acs_county[acs_county['GEOID'].isin(all_study_geoids)].copy()
        for col in ACS_VARS:
            acs_county[col] = pd.to_numeric(acs_county[col], errors='coerce')
        print(f"\nACS county data: {len(acs_county)} study counties")
    else:
        acs_county = None
        print("No ACS data retrieved")
else:
    acs_county = None
    print("No CENSUS_API_KEY — will use county_population.csv + EES as proxy")


  Fetching ACS for WY (56)... 

23 counties
  Fetching ACS for CO (08)... 

64 counties
  Fetching ACS for MT (30)... 

56 counties
  Fetching ACS for UT (49)... 

29 counties
  Fetching ACS for ID (16)... 

44 counties
  Fetching ACS for NE (31)... 

93 counties
  Fetching ACS for SD (46)... 

66 counties

ACS county data: 157 study counties


In [18]:

# 4b. Build demographic dict per county (acs or proxy)
county_pop_csv = pd.read_csv(RAW / 'census_counties/county_population.csv',
                              dtype={'GEOID': str})
county_pop_csv['GEOID'] = county_pop_csv['GEOID'].str.zfill(5)
pop_map = county_pop_csv.set_index('GEOID')['population'].to_dict()

demo = {}  # geoid → dict

if acs_county is not None:
    for _, row in acs_county.iterrows():
        geoid = str(row['GEOID']).zfill(5)
        pop = row.get('B01003_001E', None)
        mhi = row.get('B19013_001E', None)
        pci = row.get('B19301_001E', None)
        emp = row.get('B23025_004E', None)
        lf  = row.get('B23025_003E', None)
        demo[geoid] = {
            'population':              int(pop) if pd.notna(pop) else None,
            'median_household_income': int(mhi) if pd.notna(mhi) and mhi > 0 else None,
            'per_capita_income':       int(pci) if pd.notna(pci) and pci > 0 else None,
            'employment':              int(emp) if pd.notna(emp) else None,
            'source_demographics':    'acs_2022_5yr',
        }
else:
    # ACS proxy: use population CSV + EES Ec_income and employment_rate proxies
    for geoid in all_study_geoids:
        ees_row = county_ees[county_ees['geoid'] == geoid]
        pop = pop_map.get(geoid, None)
        demo[geoid] = {
            'population':              int(pop) if pop else None,
            'median_household_income': None,
            'per_capita_income':       None,
            'employment':              None,
            'source_demographics':    'acs_proxy',
        }

# Fill missing pops from county_population.csv
for geoid in all_study_geoids:
    if geoid not in demo:
        demo[geoid] = {
            'population': int(pop_map.get(geoid, 0)) or None,
            'median_household_income': None,
            'per_capita_income': None,
            'employment': None,
            'source_demographics': 'population_csv_only',
        }
    elif demo[geoid].get('population') is None:
        demo[geoid]['population'] = int(pop_map.get(geoid, 0)) or None

print(f"Demographic records assembled: {len(demo)}")
demo_sources = pd.Series({g: d['source_demographics'] for g,d in demo.items()}).value_counts()
print(demo_sources.to_string())


Demographic records assembled: 157
acs_2022_5yr    157


In [19]:

# 4c. BEA CAINC30 — employment and per_capita_income
#     Use if BEA_API_KEY is set; otherwise mark as ACS proxy.
#     BEA line codes: 10 = Per capita personal income, 7010 = employment (jobs)
if BEA_KEY:
    try:
        bea_url = "https://apps.bea.gov/api/data/"
        state_fips_list = ','.join(STUDY_STATE_FIPS.keys())
        # CAINC30 = county-level personal income and employment
        params = {
            'UserID': BEA_KEY,
            'method': 'GetData',
            'datasetname': 'Regional',
            'TableName': 'CAINC30',
            'LineCode': '10',      # per capita personal income
            'GeoFips': 'COUNTY',
            'Year': '2022',
            'ResultFormat': 'JSON',
        }
        r = requests.get(bea_url, params=params, timeout=45)
        bea_data = r.json()
        bea_df = pd.DataFrame(bea_data['BEAAPI']['Results']['Data'])
        bea_df = bea_df[bea_df['GeoFips'].isin(all_study_geoids)].copy()
        bea_df['value'] = pd.to_numeric(bea_df['DataValue'].str.replace(',',''), errors='coerce')
        for geoid in bea_df['GeoFips'].unique():
            if geoid in demo:
                v = bea_df.loc[bea_df['GeoFips']==geoid,'value'].iloc[0]
                demo[geoid]['per_capita_income'] = int(v) if pd.notna(v) else None
                demo[geoid]['source_bea'] = 'bea_cainc30_2022'
        print(f"BEA per_capita_income loaded for {len(bea_df)} counties")
    except Exception as e:
        print(f"BEA fetch failed: {e} — keeping ACS proxy")
else:
    print("No BEA_API_KEY — per_capita_income from ACS B19301_001E (source: acs_proxy where null)")
    for geoid in all_study_geoids:
        if demo.get(geoid, {}).get('per_capita_income') is None:
            demo.setdefault(geoid, {})['source_bea'] = 'acs_proxy'


No BEA_API_KEY — per_capita_income from ACS B19301_001E (source: acs_proxy where null)


In [20]:

# 4d. Generation capacity and fuel mix — spatial join of power plants to counties
pp = gpd.read_file(PROC / 'power_plants_with_ba.geojson')

# Study counties in EPSG:4326
study_poly = study_counties[['GEOID','geometry']].copy()

# Spatial join: which county polygon contains each plant
pp_joined = gpd.sjoin(
    pp[['capacity_mw','energy_source_code','technology','status','geometry']].copy(),
    study_poly.rename(columns={'GEOID':'county_geoid'}),
    how='inner', predicate='within'
)

# Active/operating plants only (status may vary)
pp_active = pp_joined[pp_joined['status'].isin(['OP','SB','OA','OS','RE']) |
                       pp_joined['status'].isna()].copy()

# Aggregate by county and fuel
fuel_agg = pp_active.groupby(['county_geoid','energy_source_code'])['capacity_mw'].sum().reset_index()
county_cap = pp_active.groupby('county_geoid')['capacity_mw'].sum().reset_index().rename(
    columns={'capacity_mw': 'total_cap_mw'})

gen_capacity = {}
fuel_mix = {}
for geoid, grp in fuel_agg.groupby('county_geoid'):
    total = grp['capacity_mw'].sum()
    gen_capacity[geoid] = round(float(total), 1)
    fuel_mix[geoid] = {
        row['energy_source_code']: round(float(row['capacity_mw'] / total), 4)
        for _, row in grp.iterrows()
    }

print(f"Counties with generation data: {len(gen_capacity)}")
# Counties with no plants get zeros
for geoid in all_study_geoids:
    if geoid not in gen_capacity:
        gen_capacity[geoid] = 0.0
        fuel_mix[geoid] = {}


Counties with generation data: 74


In [21]:

# 4e. USGS 2015 county water use — total withdrawals (MGD)
#     Try USGS NWIS REST; fall back gracefully with vintage flag.
USGS_URL = (
    "https://waterdata.usgs.gov/nwis/water_use"
    "?format=rdb&wu_area=County&wu_year=2015"
    "&wu_category=ALL"
)

water_map = {}
try:
    print("Fetching USGS 2015 county water use...")
    r = requests.get(USGS_URL, timeout=60)
    if r.ok:
        # RDB format: skip comment lines starting with #, then header, then type row
        lines = [l for l in r.text.split('\n') if not l.startswith('#') and l.strip()]
        # First non-comment line is header
        header_idx = 0
        while header_idx < len(lines) and lines[header_idx].startswith('#'):
            header_idx += 1
        # Find the data lines (after the two header rows: names + types)
        data_lines = lines[header_idx:]
        water_df = pd.read_csv(
            io.StringIO('\n'.join(data_lines)),
            sep='\t', skiprows=[1], low_memory=False, na_values=['-', '']
        )
        # USGS fips is 'county_cd' (3-char) + 'state_cd' (2-char) → GEOID = state_cd + county_cd
        if 'state_cd' in water_df.columns and 'county_cd' in water_df.columns:
            water_df['GEOID'] = (water_df['state_cd'].astype(str).str.zfill(2) +
                                  water_df['county_cd'].astype(str).str.zfill(3))
            # Total withdrawals: sum freshwater + saline surface + ground
            total_cols = [c for c in water_df.columns
                          if 'total' in c.lower() and 'withdrawal' in c.lower() and 'mgd' in c.lower()]
            if total_cols:
                water_df['total_mgd'] = pd.to_numeric(water_df[total_cols[0]], errors='coerce')
            else:
                # Fall back to summing major categories
                mgd_cols = [c for c in water_df.columns
                            if c.endswith('Mgal/d') or c.endswith('mgd')]
                water_df['total_mgd'] = water_df[mgd_cols].apply(
                    pd.to_numeric, errors='coerce').sum(axis=1) if mgd_cols else np.nan
            w = water_df[water_df['GEOID'].isin(all_study_geoids)][['GEOID','total_mgd']]
            water_map = dict(zip(w['GEOID'], w['total_mgd'].round(2)))
            print(f"  USGS water use loaded for {len(water_map)} study counties (vintage 2015)")
        else:
            print("  USGS response missing expected columns; flagging as unavailable")
    else:
        print(f"  USGS fetch returned {r.status_code}; flagging as unavailable")
except Exception as e:
    print(f"  USGS fetch error: {e}; flagging as unavailable")

print(f"Counties with water data: {len(water_map)}")


Fetching USGS 2015 county water use...


  USGS response missing expected columns; flagging as unavailable
Counties with water data: 0


In [22]:

# 4f. Hand-curated Wyoming flagship assets
#     Each asset: name, geoid, type, status, capacity_or_load_mw,
#                 operational_year, source_url
FLAGSHIP_ASSETS = [
    {
        'name':               'Kemmerer Unit 1 (Natrium)',
        'geoid':              '56023',   # Lincoln County, WY
        'county_name':        'Lincoln',
        'state':              'WY',
        'type':               'nuclear',
        'status':             'under_construction',
        'capacity_or_load_mw': 345,
        'operational_year':   2031,
        'source_url':         'https://www.terrapower.com/natrium/',
        'notes':              'TerraPower/GE-Hitachi Natrium sodium-cooled fast reactor; '
                              'DOE Milestone-Based Demonstration Program. '
                              'Source URL is TerraPower press page; '
                              'see also DOE Office of Nuclear Energy announcements.',
    },
    {
        'name':               'Naughton Gas Conversion',
        'geoid':              '56023',   # Lincoln County, WY
        'county_name':        'Lincoln',
        'state':              'WY',
        'type':               'gas',
        'status':             'planned',
        'capacity_or_load_mw': None,      # capacity not publicly confirmed
        'operational_year':   2026,
        'source_url':         'needs_citation',
        'notes':              'PacifiCorp conversion of Naughton coal units to gas; '
                              'verify exact capacity and schedule in PacifiCorp IRP.',
    },
    {
        'name':               'Meta AI Data Center (Cheyenne)',
        'geoid':              '56021',   # Laramie County, WY
        'county_name':        'Laramie',
        'state':              'WY',
        'type':               'data_center',
        'status':             'under_construction',
        'capacity_or_load_mw': 100,       # IT load MW
        'operational_year':   None,
        'source_url':         'needs_citation',
        'notes':              'Meta hyperscale campus in Cheyenne WY. '
                              'Cite Meta Infrastructure announcements for source_url.',
    },
    {
        'name':               'Jade/Crusoe Campus Phase 1 (Cheyenne)',
        'geoid':              '56021',   # Laramie County, WY
        'county_name':        'Laramie',
        'state':              'WY',
        'type':               'data_center',
        'status':             'approved_2026',
        'capacity_or_load_mw': 200,       # IT load MW
        'operational_year':   2026,
        'source_url':         'needs_citation',
        'notes':              'Jade/Crusoe AI computing campus; Laramie County approval 2026. '
                              'Cite Crusoe Energy / Jade Data Centers press release.',
    },
    {
        'name':               'BWXT TRISO Fuel Facility',
        'geoid':              '56005',   # Campbell County, WY
        'county_name':        'Campbell',
        'state':              'WY',
        'type':               'nuclear_fuel',
        'status':             'operating',
        'capacity_or_load_mw': None,
        'operational_year':   None,
        'source_url':         'needs_citation',
        'notes':              'BWXT produces TRISO fuel pellets; Gillette WY facility. '
                              'Cite BWXT press release or DOE HALEU announcement for source_url.',
    },
    {
        'name':               'Powder River Basin Coal Mines',
        'geoid':              '56005',   # Campbell County, WY
        'county_name':        'Campbell',
        'state':              'WY',
        'type':               'coal',
        'status':             'operating',
        'capacity_or_load_mw': None,      # aggregate mine output varies
        'operational_year':   None,
        'source_url':         'https://www.eia.gov/coal/data.php',
        'notes':              'Powder River Basin (PRB) is the largest US coal-producing '
                              'region; Campbell County anchors production. '
                              'See EIA coal production data for annual output.',
    },
    {
        'name':               'Jim Bridger Power Plant',
        'geoid':              '56037',   # Sweetwater County, WY
        'county_name':        'Sweetwater',
        'state':              'WY',
        'type':               'coal',
        'status':             'operating',
        'capacity_or_load_mw': 2120,      # 4 units ~530 MW each; verify against EIA-860
        'operational_year':   1974,
        'source_url':         'https://www.eia.gov/electricity/data/eia860/',
        'notes':              'PacifiCorp / Pacific Power. Units 3-4 targeted for early '
                              'retirement under PacifiCorp IRP; verify current status.',
    },
    {
        'name':               'Dave Johnston Power Plant',
        'geoid':              '56009',   # Converse County, WY
        'county_name':        'Converse',
        'state':              'WY',
        'type':               'coal',
        'status':             'operating',
        'capacity_or_load_mw': 762,       # from EIA-860; verify
        'operational_year':   1959,
        'source_url':         'https://www.eia.gov/electricity/data/eia860/',
        'notes':              'PacifiCorp / Pacific Power; Glenrock WY. '
                              'Capacity verified from EIA-860 2024.',
    },
]

# Validate: all 8 assets present with source_url or flagged
for a in FLAGSHIP_ASSETS:
    assert 'source_url' in a, f"Missing source_url for {a['name']}"
    assert 'geoid' in a, f"Missing geoid for {a['name']}"
    assert 'type' in a, f"Missing type for {a['name']}"
    assert 'status' in a, f"Missing status for {a['name']}"

print(f"✓ {len(FLAGSHIP_ASSETS)} flagship assets defined and validated")
for a in FLAGSHIP_ASSETS:
    url_flag = '⚠ needs_citation' if a['source_url'] == 'needs_citation' else '✓'
    print(f"  {url_flag}  {a['name']} ({a['state']}, {a['type']}, {a['status']})")


✓ 8 flagship assets defined and validated
  ✓  Kemmerer Unit 1 (Natrium) (WY, nuclear, under_construction)
  ⚠ needs_citation  Naughton Gas Conversion (WY, gas, planned)
  ⚠ needs_citation  Meta AI Data Center (Cheyenne) (WY, data_center, under_construction)
  ⚠ needs_citation  Jade/Crusoe Campus Phase 1 (Cheyenne) (WY, data_center, approved_2026)
  ⚠ needs_citation  BWXT TRISO Fuel Facility (WY, nuclear_fuel, operating)
  ✓  Powder River Basin Coal Mines (WY, coal, operating)
  ✓  Jim Bridger Power Plant (WY, coal, operating)
  ✓  Dave Johnston Power Plant (WY, coal, operating)


In [23]:

# 4g. Assemble county_cards.json
cards = {}

# Group flagship assets by geoid
flagship_by_county = {}
for a in FLAGSHIP_ASSETS:
    flagship_by_county.setdefault(a['geoid'], []).append(a)

for geoid in sorted(all_study_geoids):
    d = demo.get(geoid, {})
    card = {
        'geoid':                   geoid,
        'county_name':             name_map.get(geoid, ''),
        'state':                   state_map.get(geoid, ''),
        # Demographics
        'population':              d.get('population'),
        'median_household_income': d.get('median_household_income'),
        'per_capita_income':       d.get('per_capita_income'),
        'employment':              d.get('employment'),
        'source_demographics':     d.get('source_demographics', 'unknown'),
        'source_employment':       d.get('source_bea', 'acs_proxy'),
        # Generation
        'generation_capacity_mw':  gen_capacity.get(geoid, 0.0),
        'fuel_mix':                fuel_mix.get(geoid, {}),
        # Water
        'water_withdrawals_mgd':   water_map.get(geoid),
        'water_vintage':           2015 if geoid in water_map else None,
        # EES
        'E':  float(county_ees.loc[county_ees['geoid']==geoid,'E'].iloc[0])
              if geoid in county_ees['geoid'].values else None,
        'Ec': float(county_ees.loc[county_ees['geoid']==geoid,'Ec'].iloc[0])
              if geoid in county_ees['geoid'].values else None,
        'S':  float(county_ees.loc[county_ees['geoid']==geoid,'S'].iloc[0])
              if geoid in county_ees['geoid'].values else None,
        # Flagship assets
        'flagship_assets': flagship_by_county.get(geoid, []),
    }
    cards[geoid] = card

with open(PROC / 'mw_county_cards.json', 'w') as f:
    json.dump(cards, f, indent=2, default=str)

print(f"Saved mw_county_cards.json — {len(cards)} county records")
# Spot-check WY counties with flagship assets
wy_flagships = [(g, [a['name'] for a in cards[g]['flagship_assets']])
                for g in cards if cards[g]['state'] == 'WY' and cards[g]['flagship_assets']]
print(f"\nWY counties with flagship assets: {len(wy_flagships)}")
for geoid, names in sorted(wy_flagships):
    print(f"  {geoid} {cards[geoid]['county_name']}: {names}")

# Verify all 8 assets present
all_asset_names = [a['name'] for g in cards for a in cards[g]['flagship_assets']]
assert len(all_asset_names) == 8, f"Expected 8 flagship assets, found {len(all_asset_names)}"
print(f"\n✓ All 8 flagship assets present in mw_county_cards.json")


Saved mw_county_cards.json — 157 county records

WY counties with flagship assets: 5
  56005 Campbell: ['BWXT TRISO Fuel Facility', 'Powder River Basin Coal Mines']
  56009 Converse: ['Dave Johnston Power Plant']
  56021 Laramie: ['Meta AI Data Center (Cheyenne)', 'Jade/Crusoe Campus Phase 1 (Cheyenne)']
  56023 Lincoln: ['Kemmerer Unit 1 (Natrium)', 'Naughton Gas Conversion']
  56037 Sweetwater: ['Jim Bridger Power Plant']

✓ All 8 flagship assets present in mw_county_cards.json


---
## 5 · Metadata Update

Append `county_pivot` block to `network_metadata.json`.


In [24]:

# 5a. Compute crosswalk stats
n_study       = len(all_study_geoids)
total_rows    = len(crosswalk)
buses_per_county = (
    crosswalk.drop_duplicates(['geoid','bus_id'])
             .groupby('geoid')['bus_id'].nunique()
)
n_multi       = int((buses_per_county > 1).sum())
mean_buses    = float(buses_per_county.mean())

county_pivot = {
    'county_pivot': {
        'study_county_count': n_study,
        'crosswalk_stats': {
            'total_rows':                  total_rows,
            'counties_with_multiple_buses': n_multi,
            'mean_buses_per_county':        round(mean_buses, 3),
        },
        'doctrine': (
            'The county is the unit of analysis and governance. '
            'The bus is the unit of energy system intervention. '
            'The tract is the unit of social measurement. '
            'The ecoregion is the unit of ecological suitability. '
            'The material ledger is the unit of honesty.'
        ),
        'timestamp': pd.Timestamp.utcnow().isoformat(),
    }
}

print("county_pivot block:")
print(json.dumps(county_pivot, indent=2))


county_pivot block:
{
  "county_pivot": {
    "study_county_count": 157,
    "crosswalk_stats": {
      "total_rows": 726,
      "counties_with_multiple_buses": 157,
      "mean_buses_per_county": 3.0
    },
    "doctrine": "The county is the unit of analysis and governance. The bus is the unit of energy system intervention. The tract is the unit of social measurement. The ecoregion is the unit of ecological suitability. The material ledger is the unit of honesty.",
    "timestamp": "2026-06-10T13:20:51.864286+00:00"
  }
}


In [25]:

# 5b. Append to network_metadata.json
with open(PROC / 'network_metadata.json') as f:
    meta = json.load(f)

meta.update(county_pivot)

with open(PROC / 'network_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

# Verify
with open(PROC / 'network_metadata.json') as f:
    meta_check = json.load(f)

assert 'county_pivot' in meta_check, "county_pivot block missing from metadata"
assert meta_check['county_pivot']['study_county_count'] == n_study
print(f"✓ network_metadata.json updated — county_pivot block present")
print(f"  study_county_count:          {meta_check['county_pivot']['study_county_count']}")
print(f"  crosswalk total_rows:        {meta_check['county_pivot']['crosswalk_stats']['total_rows']}")
print(f"  mean_buses_per_county:       {meta_check['county_pivot']['crosswalk_stats']['mean_buses_per_county']}")


✓ network_metadata.json updated — county_pivot block present
  study_county_count:          157
  crosswalk total_rows:        726
  mean_buses_per_county:       3.0


---
## Handoff Checklist


In [26]:

print("=" * 65)
print("SESSION 0.1 HANDOFF CHECKLIST")
print("=" * 65)

checks = []

# 1. Crosswalk validation
try:
    xw = pd.read_parquet(PROC / 'county_crosswalk.parquet')
    ws = xw.drop_duplicates(['geoid','bus_id']).groupby('geoid')['bus_weight'].sum()
    bad = ws[np.abs(ws - 1.0) > 1e-6]
    assert len(bad) == 0
    checks.append(("✓", "Crosswalk validation assertions passed"))
except Exception as e:
    checks.append(("✗", f"Crosswalk validation FAILED: {e}"))

# 2. EES comparison printed (done above; assert file exists)
ees_csv = PROC / 'mw_county_ees_summary.csv'
if ees_csv.exists():
    checks.append(("✓", "EES comparison table printed; mw_county_ees_summary.csv saved"))
else:
    checks.append(("✗", "mw_county_ees_summary.csv missing"))

# 3. mw_counties.geojson ≤ 50 KB per state
counties_gj = gpd.read_file(PROC / 'mw_counties.geojson')
import json as _json
state_sizes = {}
for state, grp in counties_gj.groupby('state'):
    kb = len(grp.to_json().encode()) / 1024
    state_sizes[state] = kb
over50 = {s: kb for s, kb in state_sizes.items() if kb > 50}
if not over50:
    checks.append(("✓", f"mw_counties.geojson: all states ≤50 KB per state"))
    for s, kb in sorted(state_sizes.items()):
        print(f"     {s}: {kb:.1f} KB")
else:
    checks.append(("✗", f"States over 50KB: {over50}"))

# 4. All 8 WY flagship assets
with open(PROC / 'mw_county_cards.json') as f:
    cards_check = json.load(f)
all_assets = [a for g in cards_check for a in cards_check[g]['flagship_assets']]
if len(all_assets) == 8:
    missing_cite = [a['name'] for a in all_assets if a.get('source_url') == 'needs_citation']
    if missing_cite:
        checks.append(("⚠", f"8 assets present; {len(missing_cite)} need citation: {missing_cite}"))
    else:
        checks.append(("✓", "All 8 WY flagship assets present with source_url"))
else:
    checks.append(("✗", f"Expected 8 flagship assets, found {len(all_assets)}"))

# 5. network_metadata.json county_pivot
with open(PROC / 'network_metadata.json') as f:
    meta_final = json.load(f)
if 'county_pivot' in meta_final:
    checks.append(("✓", "network_metadata.json updated with county_pivot block"))
else:
    checks.append(("✗", "county_pivot block missing from network_metadata.json"))

print()
for flag, msg in checks:
    print(f"  [{flag}] {msg}")
print()
n_pass = sum(1 for f, _ in checks if f == "✓")
n_warn = sum(1 for f, _ in checks if f == "⚠")
n_fail = sum(1 for f, _ in checks if f == "✗")
print(f"RESULT: {n_pass} passed, {n_warn} warnings, {n_fail} failed")
if n_fail == 0:
    print("Session 0.1 ready for handoff to Session 0.2 (Action Library v3).")


SESSION 0.1 HANDOFF CHECKLIST
     CO: 41.0 KB
     ID: 9.8 KB
     MT: 49.5 KB
     NE: 5.3 KB
     SD: 11.6 KB
     UT: 12.1 KB
     WY: 17.4 KB

  [✓] Crosswalk validation assertions passed
  [✓] EES comparison table printed; mw_county_ees_summary.csv saved
  [✓] mw_counties.geojson: all states ≤50 KB per state
  [⚠] 8 assets present; 4 need citation: ['BWXT TRISO Fuel Facility', 'Meta AI Data Center (Cheyenne)', 'Jade/Crusoe Campus Phase 1 (Cheyenne)', 'Naughton Gas Conversion']
  [✓] network_metadata.json updated with county_pivot block

RESULT: 4 passed, 1 warnings, 0 failed
Session 0.1 ready for handoff to Session 0.2 (Action Library v3).
